# 자금세탁소 — 통합 모델 파이프라인 (2026-08-31)

**이 노트북 하나로 1차 9-Class 부터 2차 이진, 알림 결합, 평가까지 전부 돈다.**
`train9/`·`prep9/`·`final_9class.py` 등 외부 프로젝트 모듈을 import 하지 않는 자립형이다.
필요한 정의를 전부 이 안에 담았으므로, 노트북과 전처리 산출물만 있으면 재현된다.

## 무엇을 합치고 무엇을 안 합쳤나

08-31 세션에서 스크립트 여러 개로 흩어져 있던 **모델 경로**를 하나로 모았다.
1차 학습 → 운영점 선택 → 1차 평가 → 2차 이진 머리 → 구조 결합 → 최종 평가가 위에서 아래로 한 줄이다.

합치지 않은 것: 불균형 탐색·클래스 ablation·경계 purging·문헌 대조는 **모델 자체가 아니라 실험**이라
여기 넣지 않았다(각각 `imbalance_search.py`, `ablate_classes_blocks.py`,
`boundary_and_purging.py`, `lit_compare_9class.py`).

## 실행 방법

셀을 위에서부터 순서대로 실행한다. `RUN` 의 스위치가 실행 비용을 정한다.

| 스위치 | False (기본) | True |
|---|---|---|
| `REFIT_STAGE1` | 저장된 확률(.npy)을 읽는다 — 수 초 | 1차 모델 재학습 — **수십 분** |
| `REFIT_BINARY` | 저장된 확률(.npy)을 읽는다 — 수 초 | 이진 머리 2벌 재학습 — **약 2시간**(L_all 4,408s + L_oop 3,136s, 실측) |

기본값으로 열면 저장된 산출물을 읽어 전체 흐름과 수치를 몇 초 만에 확인할 수 있다.
처음부터 다시 학습하려면 스위치를 True 로 바꾼다.

## 반드시 지킬 규율

- **운영점(임계값·예산 배분)은 val 로만 고른다.** test 로 고르면 그 수치는 보고에 못 쓴다.
- **채점은 사전확률 보정 확률로 한다.** 보정을 빼면 `tau` 가 맞지 않는다 — 보정과 tau 는 한 묶음이다.
- **직렬 vs 병렬은 아직 팀 미결이다.** 이 노트북은 네 구조를 나란히 재기만 하고 승자를 고르지 않는다.
  (08-31 보고서 §한계: 병렬이 중복 제거 후 예산을 다 못 채워 불리했던 공정성 문제가 남아 있다.)

## 0. 설정

In [1]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, confusion_matrix

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

W       = Path('/workspace')
PROC    = W / 'processed_9class'
M9      = W / 'model_9class'          # 1차 산출물
MB      = W / 'model_binary'          # 2차 이진 머리 산출물
OUT     = W / 'model_pipeline'        # 이 노트북의 산출물
OUT.mkdir(exist_ok=True)

RUN = {
    'BASIS':         '10day',   # 주 실험 기준(꼬리 제외). 논문 대조용은 '18day'
    'REFIT_STAGE1':  False,     # True 면 1차 재학습(수십 분)
    'REFIT_BINARY':  False,     # True 면 이진 머리 재학습(약 2시간)
    'TARGET_RECALL': 0.70,      # 팀 비교 관례 — 재현율을 고정하고 정밀도를 견준다
    'TARGET_PREC':   0.90,      # 팀 정량 목표
    'N_JOBS':        24,
    'SEED':          42,
}

# 1차 설정은 08-27 에 val 로 확정해 얼린 값을 그대로 쓴다(여기서 다시 고르지 않는다).
CFG = json.loads((M9 / 'selected_config.json').read_text(encoding='utf-8'))['config']
CFG = {k: v for k, v in CFG.items() if v is not None}
assert CFG['feature_set'] == 'no_abs', '화두 10·17 — 절대 시각 피처는 모델 입력에 넣지 않는다'

CLASS_NAMES = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER',
               'BIPARTITE', 'STACK', 'RANDOM', 'OUT_OF_PATTERN']
CLASS8 = 8

print('기준:', RUN['BASIS'], '| 1차 설정:', CFG)
print('재학습:', {k: v for k, v in RUN.items() if k.startswith('REFIT')})

기준: 10day | 1차 설정: {'variant': 'full', 'feature_set': 'no_abs', 'family': 'hier_et', 'class_weight': 'balanced', 'model_kw': {'n_estimators': 200}}
재학습: {'REFIT_STAGE1': False, 'REFIT_BINARY': False}


## 1. 데이터 — 전처리 산출물 로더

`processed_9class/HI-Small_{basis}/` 를 그대로 읽는다. 여기서 하는 판단은 두 가지뿐이다.

1. 어떤 피처 열을 쓰는가 (`no_abs` = 절대 시각 제외)
2. train 표본을 어떤 변형으로 쓰는가 (`full` = 전량 + 클래스 가중)

**val·test 는 어떤 경우에도 손대지 않는다.** 평가 분포가 바뀌면 실험 간 비교가 성립하지 않는다.

In [2]:
class Basis:
    """한 기준(10day / 18day)의 전처리 산출물 묶음."""

    def __init__(self, name):
        self.name = name
        self.dir = PROC / f'HI-Small_{name}'
        self.meta = json.loads((self.dir / 'features_meta.json').read_text(encoding='utf-8'))
        self.feat_names = list(self.meta['features']['names'])

    def cols(self, feature_set):
        n = self.meta['features']['n']
        if feature_set == 'all81':
            return np.arange(n)
        if feature_set == 'no_abs':
            return np.setdiff1d(np.arange(n), list(self.meta['features']['absolute_time_cols']))
        raise ValueError(feature_set)

    # X 는 메모리에 통째로 올리지 않는다(수백만 행 × 81열)
    def X(self, tag):        return np.load(self.dir / 'stage1_9class' / f'X_{tag}.npy', mmap_mode='r')
    def y(self, tag):        return np.load(self.dir / 'stage1_9class' / f'y9_{tag}.npy')
    def is_pos(self, tag):   return np.load(self.dir / 'stage1_9class' / f'is_pos_{tag}.npy').astype(bool)
    def is_oop(self, tag):   return np.load(self.dir / 'stage1_9class' / f'is_oop_{tag}.npy').astype(bool)
    def index_full(self):    return np.load(self.dir / 'index_full.npz')

    def sample_idx(self, variant):
        """train 내부 인덱스. 'full' 이면 None(전량 사용)."""
        if variant == 'full':
            return None
        return np.load(self.dir / 'undersample' / f'{variant}_tr.npy')


def build_train(b, variant, feature_set, class_weight='balanced'):
    """학습 표본 + 표본가중 + '모델이 본 사전확률'을 함께 돌려준다."""
    cols = b.cols(feature_set)
    y_full = b.y('tr')
    idx = b.sample_idx(variant)
    Xmm = b.X('tr')
    if idx is None:
        X, y = np.asarray(Xmm[:, cols]), y_full
    else:
        X, y = np.asarray(Xmm[idx])[:, cols], y_full[idx]

    cnt = np.bincount(y, minlength=9).astype(np.float64)
    if class_weight == 'balanced':
        w_cls = np.where(cnt > 0, len(y) / (9.0 * np.maximum(cnt, 1.0)), 0.0)
    elif class_weight == 'none':
        w_cls = np.ones(9)
    else:
        raise ValueError(class_weight)
    w = w_cls[y]

    wsum = np.array([w[y == c].sum() for c in range(9)], dtype=np.float64)
    prior_model = wsum / wsum.sum()                       # 모델이 본 사전확률
    cnt_full = np.bincount(y_full, minlength=9).astype(np.float64)
    prior_true = cnt_full / cnt_full.sum()                # 참 사전확률은 train 에서만 잰다
    return {'X': X, 'y': y, 'w': w, 'cols': cols,
            'prior_model': prior_model, 'prior_true': prior_true, 'n_rows': int(len(y))}


def predict_proba_chunked(model, b, tag, cols, n_out=9, step=400_000):
    """전량 예측. 피처 행렬을 통째로 RAM 에 올리지 않는다."""
    X = b.X(tag); n = X.shape[0]
    out = np.empty((n, n_out) if n_out > 1 else n, dtype=np.float32)
    for lo in range(0, n, step):
        blk = np.asarray(X[lo:lo + step])[:, cols]
        p = model.predict_proba(blk)
        out[lo:lo + step] = p.astype(np.float32) if n_out > 1 else p[:, 1].astype(np.float32)
    return out


def apply_prior_correction(proba, prior_model, prior_true):
    """p_true(c|x) ∝ p_model(c|x) · π_true(c) / π_model(c) — 행별 재정규화.

    class_weight='balanced' 때문에 prior_model 이 균등(1/9)이라, 실제로는 클래스별 상수
    재척도이자 p_c/p_8 오즈 순위로 작동한다. 같은 임계 규칙에서 원 점수보다 정밀도가 높다.
    """
    f = np.where(prior_model > 0, prior_true / np.maximum(prior_model, 1e-12), 0.0)
    p = proba * f[None, :]
    return np.divide(p, np.maximum(p.sum(axis=1, keepdims=True), 1e-12))


b = Basis(RUN['BASIS'])
idx_full = b.index_full()
print(f"{b.name} · 피처 {b.meta['features']['n']}열 (사용 {len(b.cols(CFG['feature_set']))}열)")
for t in ('tr', 'va', 'te'):
    y_ = b.y(t)
    print(f'  {t}: {len(y_):>9,}행 · 패턴 {int((y_ <= 7).sum()):>5,} · 세탁 {int(b.is_pos(t).sum()):>5,}')

10day · 피처 81열 (사용 76열)
  tr: 3,045,987행 · 패턴 1,176 · 세탁 2,296
  va: 1,015,195행 · 패턴   694 · 세탁 1,083
  te: 1,015,657행 · 패턴   684 · 세탁 1,143


## 2. 모델 — 계층형 9-Class

평탄한 9-Class 는 1,176건뿐인 패턴 행을 8개 클래스로 다시 쪼개 학습한다. 가장 작은 클래스가
train 60건이라 어떤 모델도 안정적으로 못 배운다. 그래서 두 단으로 쪼개되 **출력 계약은 9-Class 그대로** 둔다.

- (a) 이진 머리: '패턴인가 아닌가' — 1,176건을 한 덩어리로 쓴다
- (b) 유형 머리: '어느 유형인가' — 패턴 행만 보므로 불균형이 사라진다

`p(c|x) = p_bin(패턴|x) · p_type(c|x)` (c=0..7), `p(8|x) = 1 − p_bin(패턴|x)`

In [3]:
def make_model(name, n_jobs=None, seed=None, **over):
    n_jobs = RUN['N_JOBS'] if n_jobs is None else n_jobs
    seed = RUN['SEED'] if seed is None else seed
    if name == 'et':
        kw = dict(n_estimators=600, max_features='sqrt', min_samples_leaf=1,
                  n_jobs=n_jobs, random_state=seed)
        kw.update(over)
        return ExtraTreesClassifier(**kw)
    if name.startswith('hier_'):
        return HierarchicalNine(family=name.split('_', 1)[1], n_jobs=n_jobs, seed=seed, **over)
    raise ValueError(f'이 노트북은 et / hier_et 만 싣는다: {name}')


class HierarchicalNine:
    """9-Class 를 두 단으로 쪼개 학습하되 출력은 9열 확률 그대로."""

    def __init__(self, family='et', n_jobs=24, seed=42, **over):
        self.family, self.n_jobs, self.seed, self.over = family, n_jobs, seed, over
        self.bin_ = self.type_ = self.type_classes_ = None
        self.classes_ = list(range(9))

    def fit(self, X, y, sample_weight=None):
        y = np.asarray(y)
        # (a) 패턴 vs 클래스8 — 가중을 두 덩어리 기준으로 다시 잡는다
        yb = (y <= 7).astype(np.int8)
        cb = np.bincount(yb, minlength=2).astype(float)
        wb = np.where(cb > 0, len(yb) / (2.0 * np.maximum(cb, 1.0)), 0.0)[yb]
        self.bin_ = make_model(self.family, self.n_jobs, self.seed, **self.over)
        self.bin_.fit(X, yb, sample_weight=wb)

        # (b) 유형 8-way — 패턴 행만, 8클래스 기준 균형 가중
        m = y <= 7
        yt = y[m]
        ct = np.bincount(yt, minlength=8).astype(float)
        wt = np.where(ct > 0, m.sum() / (8.0 * np.maximum(ct, 1.0)), 0.0)[yt]
        self.type_ = make_model(self.family, self.n_jobs, self.seed, **self.over)
        self.type_.fit(X[m], yt, sample_weight=wt)
        self.type_classes_ = np.asarray(self.type_.classes_)
        return self

    def predict_proba(self, X):
        i1 = int(np.where(np.asarray(self.bin_.classes_) == 1)[0][0])
        p_pat = self.bin_.predict_proba(X)[:, i1]
        pt = self.type_.predict_proba(X)
        out = np.zeros((X.shape[0], 9), dtype=np.float64)
        for j, c in enumerate(self.type_classes_):
            out[:, int(c)] = p_pat * pt[:, j]
        out[:, 8] = 1.0 - p_pat
        return out

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

## 3. 지표

정확도는 주지표가 아니다 — 전체의 99.95%가 클래스 8이라 **전부 8이라고 답해도 99.9%** 다.
대신 파이프라인이 실제로 하는 두 가지 일을 따로 잰다.

1. **즉시 알림** — 0~7로 예측한 거래는 그 자리에서 알림이 된다(정밀도가 중요)
2. **2차 라우팅** — 8로 예측한 거래만 2차로 넘어간다(여기서 놓치면 영영 못 본다)

정밀도는 셋을 구분한다. 뭉뚱그리면 "유형은 틀렸지만 세탁은 맞은" 알림이 성공인지 실패인지 말할 수 없다.
`type`(유형까지 정확) · `any`(8종 패턴이긴 함) · `laundering`(세탁이긴 함).

In [4]:
def _prf(y_true, y_pred, c):
    tp = int(((y_pred == c) & (y_true == c)).sum())
    fp = int(((y_pred == c) & (y_true != c)).sum())
    fn = int(((y_pred != c) & (y_true == c)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0), tp + fn


def per_class_table(y_true, y_pred, proba=None):
    rows = []
    for c in range(9):
        p, r, f, n = _prf(y_true, y_pred, c)
        row = {'class': c, 'name': CLASS_NAMES[c], 'support': n, 'precision': p,
               'recall': r, 'f1': f, 'n_pred': int((y_pred == c).sum())}
        if proba is not None and 0 < n < len(y_true):
            row['pr_auc'] = float(average_precision_score((y_true == c).astype(int), proba[:, c]))
        rows.append(row)
    return pd.DataFrame(rows)


def evaluate(y_true, proba, is_pos, is_oop, tau=None):
    """argmax9(tau=None) 또는 임계 규칙(tau)으로 판정하고 지표를 낸다."""
    score = proba[:, :8].max(axis=1)
    arg = proba[:, :8].argmax(axis=1)
    y_pred = proba.argmax(axis=1) if tau is None else np.where(score >= tau, arg, CLASS8)
    alert = y_pred <= 7
    true_pat = y_true <= 7
    n_alert = int(alert.sum())
    per = per_class_table(y_true, y_pred, proba)
    # macro 는 평가셋에 실제로 등장한 클래스만 평균한다(분모가 표마다 달라지면 비교 불가)
    pr = per[(per['class'] <= 7) & (per['support'] > 0)]
    return {
        'tau': tau, 'n_rows': int(len(y_true)),
        'macro_recall_8': float(pr['recall'].mean()),
        'macro_precision_8': float(pr['precision'].mean()),
        'macro_f1_8': float(pr['f1'].mean()),
        'macro_pr_auc_8': float(pr['pr_auc'].mean()) if 'pr_auc' in pr else np.nan,
        'n_alerts': n_alert,
        'alert_precision_type': float((y_pred[alert] == y_true[alert]).mean()) if n_alert else 0.0,
        'alert_precision_any': float(true_pat[alert].mean()) if n_alert else 0.0,
        'alert_precision_laundering': float(is_pos[alert].mean()) if n_alert else 0.0,
        'alert_recall_pattern': float(alert[true_pat].mean()) if true_pat.any() else 0.0,
        'routing_recall_class8': float((y_pred[y_true == CLASS8] == CLASS8).mean()),
        'oop_routed_to_stage2': float((y_pred[is_oop] == CLASS8).mean()) if is_oop.any() else np.nan,
        'normal_false_alert_rate': float(alert[(y_true == CLASS8) & (~is_pos)].mean()),
        'laundering_alerted': float(alert[is_pos].mean()) if is_pos.any() else np.nan,
    }


def alert_budget_sweep(y_true, proba, is_pos, min_k=20, max_k=200_000):
    """알림 예산 K 별 정밀도-재현율 곡선.

    임계값을 분위수로 자르지 않는다 — 양성이 100만 행 중 700건 수준이라 분위수 격자는
    고정밀 구간(상위 수백 건)을 통째로 건너뛴다. 팀 목표인 정밀도 90%가 바로 그 구간에 있다.
    K 는 그대로 '관제팀이 하루에 볼 수 있는 알림 수'라 운영 대화에도 쓰인다.
    """
    score = proba[:, :8].max(axis=1)
    arg = proba[:, :8].argmax(axis=1)
    order = np.argsort(-score, kind='stable')
    true_pat = y_true <= 7
    n_pat = int(true_pat.sum())
    yo, ao, po = y_true[order], arg[order], is_pos[order]
    c_any, c_type, c_laund = np.cumsum(true_pat[order]), np.cumsum(ao == yo), np.cumsum(po)
    c_cls = np.stack([np.cumsum((ao == c) & (yo == c)) for c in range(8)])
    sup = np.array([int((y_true == c).sum()) for c in range(8)])

    hi = int(min(max_k, len(y_true)))
    ks = np.unique(np.round(np.logspace(np.log10(min_k), np.log10(hi), 45)).astype(int))
    rows = []
    for k in ks:
        k = int(min(k, len(y_true)))
        if k < min_k:
            continue
        i = k - 1
        rec = np.where(sup > 0, c_cls[:, i] / np.maximum(sup, 1), np.nan)
        rows.append({'k_alerts': k, 'tau': float(score[order[i]]),
                     'precision_type': float(c_type[i] / k),
                     'precision_any': float(c_any[i] / k),
                     'precision_laundering': float(c_laund[i] / k),
                     'recall_pattern': float(c_any[i] / max(n_pat, 1)),
                     'macro_recall_8': float(np.nanmean(rec))})
    return pd.DataFrame(rows)


def op_at_recall(sw, target=0.70):
    """재현율을 고정해 놓고 그 지점의 정밀도를 본다 — 팀 비교 관례.

    "recall 0.7 이상"이 넘어야 할 기준선이 아니라, 같은 재현율에서 어느 설정의 정밀도가
    높은지로 비교한다는 뜻이다. 목표를 만족하는 가장 작은 예산을 고른다.
    """
    ok = sw[sw['recall_pattern'] >= target]
    return None if not len(ok) else ok.loc[ok['k_alerts'].idxmin()].to_dict()


def op_at_precision(sw, target=0.90, field='precision_laundering'):
    """목표 정밀도를 만족하는 운영점 중 재현율이 가장 큰 것."""
    ok = sw[sw[field] >= target]
    return None if not len(ok) else ok.loc[ok['recall_pattern'].idxmax()].to_dict()


def confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(9)))
    return pd.DataFrame(cm, index=[f'true {n}' for n in CLASS_NAMES],
                        columns=[f'pred {n}' for n in CLASS_NAMES])

## 4. 후처리 — 거래 알림을 블록으로 묶기

회의 결정 2·3. 모델은 거래별 점수만 낸다. 연관 거래를 묶는 것은 별도 후처리의 몫이다.

묶는 규칙: **계좌 공유 + window 분 이내**의 이행적 연결 컴포넌트.
A-B 가 창 안이고 B-C 가 창 안이면 A 와 C 가 멀어도 한 블록이다.
이 함수들은 라벨을 쓰지 않으므로 운영 추론 시점에 그대로 돌릴 수 있다.

In [5]:
def link_components(src, dst, ts, window_min):
    """거래들을 '계좌 공유 + window 분 이내'로 이어붙인 연결 컴포넌트 ID.

    엣지 하나를 (src, dst) 두 개의 (계좌, 시각) 사건으로 펼친 뒤 같은 계좌 안에서 시각순
    인접쌍만 잇는다. 완전 그래프를 만들지 않으므로 O(n log n) 이다.
    """
    m = len(src)
    if m == 0:
        return np.zeros(0, dtype=np.int32)
    ent = np.concatenate([src, dst])
    eid = np.tile(np.arange(m, dtype=np.int32), 2)
    t = np.concatenate([ts, ts])
    o = np.lexsort((t, ent))
    ent_s, eid_s, t_s = ent[o], eid[o], t[o]
    link = (ent_s[1:] == ent_s[:-1]) & ((t_s[1:] - t_s[:-1]) <= window_min)
    g = coo_matrix((np.ones(int(link.sum()), dtype=np.int8),
                    (eid_s[:-1][link], eid_s[1:][link])), shape=(m, m))
    return connected_components(g, directed=False)[1].astype(np.int32)


def _modal_class(cls_sorted, offs, n_blk):
    """블록 라벨 = 구성 거래 예측 클래스의 최빈값."""
    if n_blk == 0:
        return np.zeros(0, dtype=np.int8)
    blk = np.repeat(np.arange(n_blk), np.diff(offs))
    cnt = np.bincount(blk.astype(np.int64) * 8 + cls_sorted.astype(np.int64),
                      minlength=n_blk * 8).reshape(n_blk, 8)
    return cnt.argmax(axis=1).astype(np.int8)


def make_alerts(pred_class, pred_binary, src, dst, ts, window_min):
    """거래별 판정 -> 알림 목록. 패턴은 블록 알림, 2차 이진 적발은 단건 알림."""
    n = len(pred_class)
    row_id = np.arange(n, dtype=np.int64)
    hit = np.flatnonzero((pred_class >= 0) & (pred_class <= 7))
    comp = link_components(src[hit], dst[hit], ts[hit], window_min)
    n_blk = int(comp.max()) + 1 if len(comp) else 0
    order = np.argsort(comp, kind='stable')
    offs = np.searchsorted(comp[order], np.arange(n_blk + 1))
    pattern = {'member_rows': row_id[hit][order], 'offsets': offs, 'n_blocks': n_blk,
               'block_class': _modal_class(pred_class[hit][order], offs, n_blk),
               'window_min': window_min}
    if pred_binary is None:
        single_rows = np.zeros(0, dtype=np.int64)
    else:
        single_rows = row_id[np.flatnonzero((pred_class == 8) & (np.asarray(pred_binary) == 1))]
    return {'pattern': pattern,
            'single': {'member_rows': single_rows, 'n_blocks': len(single_rows)}}

## 5. 구조 결합 — 직렬 / 병렬 / 순위융합

**팀 미결 사항이다.** 어느 구조를 쓸지 정하지 않았으므로 여기서는 네 가지를 나란히 재기만 한다.

현재 파이프라인은 직렬(캐스케이드)이고, 팀이 제기한 손실은 두 가지다.

- **(L1)** 1차가 놓친 *패턴* 세탁을 2차가 잡을 수 없다 — 2차 학습셋이 '패턴 외 세탁'만 양성으로 보기 때문
- **(L2)** 패턴 점수가 임계선 바로 위라 통과했는데 블록으로 안 뭉치면 알림 가치가 사라진다.
  직렬에서는 그 거래가 이진 점수를 받아볼 기회가 영영 없다

공정 비교 규칙: **총 알림 예산 K 를 맞춘다**(관제팀이 하루에 볼 수 있는 건수가 실제 운영 제약이므로).
패턴 알림 몫 `kp_frac` 은 val 에서만 고른다.

In [6]:
BUDGETS   = [300, 500, 843, 1200, 2000, 3000]                    # 총 알림 예산 K
KP_FRACS  = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]  # 그중 패턴 알림 몫
WINDOW    = 60                                                    # 알림 블록 시간창(분)


def _topk_excluding(score, exclude_mask, k):
    """제외 마스크가 걸린 행을 빼고 상위 k개 — 예산을 정확히 채운다."""
    if k <= 0:
        return np.empty(0, dtype=np.int64)
    order = np.argsort(-score, kind='stable')
    return order[~exclude_mask[order]][:k]


def alerts_serial(sp, sl, kp, kl, oracle_oop=None):
    """직렬: 패턴 상위 kp 를 먼저 알림. 나머지 행만 2차로 내려보내 상위 kl 을 단건 알림."""
    ip = np.argsort(-sp, kind='stable')[:kp]
    ex = np.zeros(len(sp), bool); ex[ip] = True
    if oracle_oop is not None:
        ex |= ~oracle_oop            # 정답 기준 '패턴 외'가 아닌 행은 2단에 못 간다
    return ip, _topk_excluding(sl, ex, kl)


def alerts_parallel(sp, sl, kp, kl, oracle_oop=None):
    """병렬(합집합): 두 점수를 전 거래에 매긴 뒤 각각 상위를 뽑고 합친다.

    중복은 패턴 알림이 흡수하고 그만큼 세탁 쪽에서 다음 순위를 더 채워 총 알림 수를 맞춘다.
    """
    ip = np.argsort(-sp, kind='stable')[:kp]
    ex = np.zeros(len(sp), bool); ex[ip] = True
    return ip, _topk_excluding(sl, ex, kl)


def alerts_fusion(sp, sl, kp, kl, oracle_oop=None):
    """병렬(순위융합): 두 점수를 순위로 정규화해 가중합한 단일 점수로 상위 K 를 뽑는다.

    s_p 는 9-class 사후확률의 최댓값, s_l 은 이진 확률이라 스케일과 분포가 다르다.
    합집합과 달리 두 점수가 **모두** 높은 행을 위로 올리므로 (L2) 를 정면으로 겨냥한다.
    """
    n = len(sp); alpha = kp / max(kp + kl, 1)
    rp = np.empty(n); rp[np.argsort(-sp, kind='stable')] = np.arange(n)
    rl = np.empty(n); rl[np.argsort(-sl, kind='stable')] = np.arange(n)
    top = np.argsort(alpha * rp + (1 - alpha) * rl, kind='stable')[:(kp + kl)]
    # 유형 라벨을 붙일 수 있는 건 패턴 점수 상위 행뿐이므로 그것만 블록으로 묶는다
    is_pat = np.zeros(n, bool); is_pat[np.argsort(-sp, kind='stable')[:max(kp, 1)]] = True
    return top[is_pat[top]], top[~is_pat[top]]


def eval_alerts(ip, il, ispos, y9, src, dst, ts, att, elig):
    """거래 단위 + 블록 단위 지표. 패턴 알림만 블록으로 묶고 단건 알림은 그대로 센다."""
    al = np.concatenate([ip, il]); n = len(al)
    tp = int(ispos[al].sum())
    P = tp / max(n, 1); R = tp / max(int(ispos.sum()), 1)
    out = {'n_alerts': n, 'n_pattern': len(ip), 'n_single': len(il),
           'precision': P, 'recall': R, 'f1': 2 * P * R / max(P + R, 1e-12),
           'P_pattern_part': float(ispos[ip].mean()) if len(ip) else np.nan,
           'P_single_part': float(ispos[il].mean()) if len(il) else np.nan,
           'type_hit': float((y9[ip] <= 7).mean()) if len(ip) else np.nan}
    if len(ip):
        comp = link_components(src[ip], dst[ip], ts[ip], WINDOW)
        nb = int(comp.max()) + 1
        hit = np.zeros(nb, bool)
        for c, r in zip(comp, ip):
            hit[c] |= bool(ispos[r])
        out.update({'n_blocks': nb, 'block_precision': float(hit.mean()),
                    'alerts_per_block': round(len(ip) / max(nb, 1), 2)})
    else:
        out.update({'n_blocks': 0, 'block_precision': np.nan, 'alerts_per_block': 0.0})
    # 사건 적발률의 분모는 그 split 에 온전히 들어온 '채점 자격' 시도만이다.
    # 경계에 걸린 시도를 세면 거래 1건짜리 조각이 자동 적발로 잡혀 지표가 부풀려진다.
    if elig.any():
        alerted = np.zeros(len(att), bool); alerted[al] = True
        d = pd.DataFrame({'a': att[elig], 'h': alerted[elig]}).groupby('a')['h'].any()
        out.update({'n_attempts': int(len(d)), 'attempt_detection': float(d.mean())})
    else:
        out.update({'n_attempts': 0, 'attempt_detection': np.nan})
    return out

---
# 실행

여기서부터는 정의가 아니라 실행이다. 위 셀을 모두 실행한 뒤 순서대로 내려간다.

## 6. 1차 9-Class — 학습 또는 캐시 로드

In [7]:
t0 = time.time()
if RUN['REFIT_STAGE1']:
    tr = build_train(b, CFG['variant'], CFG['feature_set'], CFG.get('class_weight', 'balanced'))
    print(f"train {tr['n_rows']:,}행 · {len(tr['cols'])}열 — 적합 시작(수십 분)", flush=True)
    m9 = make_model(CFG['family'], **(CFG.get('model_kw') or {}))
    m9.fit(tr['X'], tr['y'], sample_weight=tr['w'])
    pv_raw = predict_proba_chunked(m9, b, 'va', tr['cols'])
    pt_raw = predict_proba_chunked(m9, b, 'te', tr['cols'])
    # 채점은 반드시 보정 확률로 한다 — 보정과 tau 는 한 묶음이다
    pv = apply_prior_correction(pv_raw, tr['prior_model'], tr['prior_true'])
    pt = apply_prior_correction(pt_raw, tr['prior_model'], tr['prior_true'])
    np.save(OUT / 'proba_val.npy', pv); np.save(OUT / 'proba_test.npy', pt)
    print(f'적합·예측 {time.time() - t0:,.0f}s')
else:
    m9 = None
    pv = np.load(M9 / 'proba_val.npy')      # 08-27 확정 모델의 저장 확률(이미 보정된 값)
    pt = np.load(M9 / 'proba_test.npy')
    print(f'캐시 로드 — 재학습하려면 RUN["REFIT_STAGE1"]=True')

yv, ipv, oopv = b.y('va'), b.is_pos('va'), b.is_oop('va')
yt, ipt, oopt = b.y('te'), b.is_pos('te'), b.is_oop('te')
assert len(pv) == len(yv) and len(pt) == len(yt), '확률 행 수가 라벨과 어긋난다 — 기준(basis) 확인'
print(f'val {pv.shape} · test {pt.shape} · {time.time() - t0:,.0f}s')

캐시 로드 — 재학습하려면 RUN["REFIT_STAGE1"]=True


val (1015195, 9) · test (1015657, 9) · 0s


## 7. 운영점 — **val 로만** 고른다

주 운영점은 팀 비교 관례를 따라 **재현율 0.70 고정** 지점이다.
팀 정량 목표(정밀도 90%) 지점도 함께 찾아 달성 가능 여부를 그대로 남긴다.

In [8]:
sw_val = alert_budget_sweep(yv, pv, ipv)
op = op_at_recall(sw_val, RUN['TARGET_RECALL'])
p90 = op_at_precision(sw_val, RUN['TARGET_PREC'], 'precision_laundering')
assert op is not None, f"val 에서 재현율 {RUN['TARGET_RECALL']} 도달 불가 — 목표를 낮추거나 모델을 바꿔야 한다"
TAU = float(op['tau'])

print(f"[운영점] val 재현율 {RUN['TARGET_RECALL']:.2f} 고정 · K={int(op['k_alerts']):,} · tau={TAU:.6g}")
print(f"         정밀도 any {op['precision_any']:.4f} · type {op['precision_type']:.4f} "
      f"· 세탁 {op['precision_laundering']:.4f} · macroR8 {op['macro_recall_8']:.4f}")
print(f"[목표] 정밀도 {RUN['TARGET_PREC']:.0%} val 달성: "
      f"{'가능' if p90 else '불가'}"
      + (f" (그때 재현율 {p90['recall_pattern']:.4f}, K={int(p90['k_alerts']):,})" if p90 else
         f" (val 최고 세탁 정밀도 {sw_val['precision_laundering'].max():.4f})"))
sw_val.head(12).round(4)

[운영점] val 재현율 0.70 고정 · K=866 · tau=2.47191e-06
         정밀도 any 0.5762 · type 0.1824 · 세탁 0.5855 · macroR8 0.1912
[목표] 정밀도 90% val 달성: 가능 (그때 재현율 0.1138, K=87)


,k_alerts,tau,precision_type,precision_any,precision_laundering,recall_pattern,macro_recall_8
0,20,0.0001,0.4000,0.9000,0.9000,0.0259,0.0113
1,25,0.0001,0.4000,0.8800,0.8800,0.0317,0.0145
2,30,0.0001,0.4000,0.9000,0.9000,0.0389,0.0166
3,37,0.0001,0.4324,0.8649,0.8649,0.0461,0.0215
4,46,0.0001,0.4565,0.8913,0.8913,0.0591,0.0261
5,57,0.0000,0.4561,0.9123,0.9123,0.0749,0.0325
6,70,0.0000,0.4143,0.9143,0.9143,0.0922,0.0355
7,87,0.0000,0.3793,0.9080,0.9080,0.1138,0.0408
8,107,0.0000,0.3551,0.8785,0.8785,0.1354,0.0477
9,132,0.0000,0.3333,0.8409,0.8409,0.1599,0.0565


## 8. 1차 test 평가 — test 는 여기서 처음 연다

In [9]:
res = {'argmax9 (참고)': evaluate(yt, pt, ipt, oopt, tau=None),
       f"운영점 R{RUN['TARGET_RECALL']:.2f}·K={int(op['k_alerts'])} (val 결정)":
           evaluate(yt, pt, ipt, oopt, tau=TAU)}
tbl = pd.DataFrame(res).T
display(tbl[['n_alerts', 'alert_precision_any', 'alert_precision_type',
             'alert_precision_laundering', 'macro_recall_8', 'macro_pr_auc_8',
             'routing_recall_class8']].round(4))

y_pred_op = np.where(pt[:, :8].max(1) >= TAU, pt[:, :8].argmax(1), CLASS8)
display(per_class_table(yt, y_pred_op, pt).round(4))

,n_alerts,alert_precision_any,alert_precision_type,alert_precision_laundering,macro_recall_8,macro_pr_auc_8,routing_recall_class8
argmax9 (참고),0.0,0.0000,0.0000,0.0000,0.0000,0.1262,1.0000
운영점 R0.70·K=866 (val 결정),843.0,0.5647,0.1447,0.5765,0.1483,0.1262,0.9996


,class,name,support,precision,recall,f1,n_pred,pr_auc
0,0,FAN-OUT,74,0.1132,0.0811,0.0945,53,0.1822
1,1,FAN-IN,71,0.3061,0.2113,0.2500,49,0.2023
2,2,CYCLE,65,0.0851,0.0615,0.0714,47,0.0512
3,3,SCATTER-GATHER,123,0.1514,0.3496,0.2113,284,0.2293
4,4,GATHER-SCATTER,147,0.1233,0.1905,0.1497,227,0.1789
5,5,BIPARTITE,52,0.1111,0.0962,0.1031,45,0.0343
6,6,STACK,107,0.1533,0.1963,0.1721,137,0.1073
7,7,RANDOM,45,0.0000,0.0000,0.0000,1,0.0243
8,8,OUT_OF_PATTERN,1014973,0.9998,0.9996,0.9997,1014814,1.0000


In [10]:
# 후처리 블록 — 거래 알림 n건이 블록 알림 m건으로 줄어든다(적발 여부는 안 바뀐다)
sel = np.flatnonzero(idx_full['split'] == 2)
src_te, dst_te, ts_te = idx_full['src_id'][sel], idx_full['dst_id'][sel], idx_full['ts_min'][sel]
att_te = idx_full['attempt'][sel]

rows = []
for wn in (b.meta['blocks']['window_chosen'], 1440, 60):
    al = make_alerts(y_pred_op, None, src_te, dst_te, ts_te, wn)
    p = al['pattern']; nb = p['n_blocks']; mem, offs = p['member_rows'], p['offsets']
    laund = np.array([bool(ipt[mem[offs[i]:offs[i + 1]]].any()) for i in range(nb)])
    rows.append({'window_min': wn, 'n_alert_tx': len(mem), 'n_blocks': nb,
                 'alerts_per_block': round(len(mem) / max(nb, 1), 2),
                 'block_precision_laundering': float(laund.mean()) if nb else np.nan})
pd.DataFrame(rows)

,window_min,n_alert_tx,n_blocks,alerts_per_block,block_precision_laundering
0,10080,843,542,1.56,0.374539
1,1440,843,561,1.50,0.393939
2,60,843,797,1.06,0.565872


## 9. 2차 이진 머리 — L_all(병렬용) / L_oop(직렬용)

왜 두 벌인가: 공정 비교를 위해서다.

- `L_oop` — 패턴 외 행(y9==8)만으로 학습. **현재 직렬 구조의 2차**. 정답 라우팅으로 만든
  학습셋이라 직렬에 유리하게 편향돼 있다. 그대로 두고 편향을 명시한다
  (직렬이 유리한 조건에서도 병렬이 이기면 결론이 더 강해진다)
- `L_all` — 전 거래로 학습. **병렬 구조의 이진 머리**. 패턴 행도 양성으로 보므로 1차가 놓친
  패턴 세탁을 잡을 수 있다. 이 차이가 병렬의 핵심 가설이다

실측 학습 시간: L_all 4,408s · L_oop 3,136s (합쳐 약 2시간).

In [11]:
def fit_binary_head(name, row_mask_tr):
    """row_mask_tr 로 고른 train 행에 이진(세탁 여부) 머리를 적합하고 val/test 확률을 낸다."""
    t = time.time()
    tr = build_train(b, CFG['variant'], CFG['feature_set'], 'balanced')   # 1차와 같은 피처·열
    cols, X = tr['cols'], tr['X']
    ypos = b.is_pos('tr').astype(np.int8)
    assert len(ypos) == len(X), f'행 수 불일치: X {len(X)} vs is_pos {len(ypos)} — variant 확인'
    m_ = np.ones(len(X), bool) if row_mask_tr is None else row_mask_tr
    Xf, yf = X[m_], ypos[m_]
    cb = np.bincount(yf, minlength=2).astype(float)
    w = np.where(cb > 0, len(yf) / (2.0 * np.maximum(cb, 1.0)), 0.0)[yf]  # 1차와 같은 balanced 가중
    mdl = make_model('et', n_estimators=400)
    mdl.fit(Xf, yf, sample_weight=w)
    pv_ = predict_proba_chunked(mdl, b, 'va', cols, n_out=1)
    pt_ = predict_proba_chunked(mdl, b, 'te', cols, n_out=1)
    np.save(OUT / f'p_{name}_val.npy', pv_); np.save(OUT / f'p_{name}_test.npy', pt_)
    print(f'[{name}] train {len(yf):,}행 · 양성 {int(yf.sum()):,} '
          f'(1:{int((len(yf) - yf.sum()) / max(yf.sum(), 1)):,}) · {time.time() - t:,.0f}s', flush=True)
    return mdl, pv_, pt_


HEADS = {}
if RUN['REFIT_BINARY']:
    y9tr = b.y('tr')
    for nm, mask in (('L_all', None), ('L_oop', y9tr == 8)):
        _, pv_, pt_ = fit_binary_head(nm, mask)
        HEADS[nm] = {'va': pv_, 'te': pt_}
else:
    for nm in ('L_all', 'L_oop'):
        HEADS[nm] = {'va': np.load(MB / f'p_{nm}_val.npy'), 'te': np.load(MB / f'p_{nm}_test.npy')}
    print('캐시 로드 — 재학습하려면 RUN["REFIT_BINARY"]=True')

print({h: {k: v.shape for k, v in d.items()} for h, d in HEADS.items()})

캐시 로드 — 재학습하려면 RUN["REFIT_BINARY"]=True
{'L_all': {'va': (1015195,), 'te': (1015657,)}, 'L_oop': {'va': (1015195,), 'te': (1015657,)}}


## 10. 구조 비교 — 직렬 vs 병렬 (예산 맞춤 2×2)

구조(직렬/병렬)와 이진 학습셋(L_oop/L_all)을 같이 바꾸면 어느 쪽이 효과를 냈는지 말할 수 없다.
네 칸을 다 재서 "구조 때문인지, 학습셋 때문인지, 둘의 상호작용인지"를 가른다.

**운영점(kp_frac)은 val 에서만 고르고, test 는 마지막에 딱 한 번 연다.**

In [12]:
el = pd.read_csv(b.dir / 'blocks' / 'attempt_eligibility.csv')
ELIG = el.loc[el['eligible'].astype(bool), 'att'].to_numpy()
print(f'채점 자격 시도 {len(ELIG)}건 / 전체 {len(el)}건')

CTX = {}
for tag, s in (('va', 1), ('te', 2)):
    m = idx_full['split'] == s
    a_ = idx_full['attempt'][m]
    CTX[tag] = dict(ispos=b.is_pos(tag), y9=b.y(tag), src=idx_full['src_id'][m],
                    dst=idx_full['dst_id'][m], ts=idx_full['ts_min'][m], att=a_,
                    elig=np.isin(a_, ELIG) & (a_ >= 0))

P9 = {'va': pv, 'te': pt}
ARCHS = (('직렬', alerts_serial, False),
         ('직렬(오라클라우팅)', alerts_serial, True),
         ('병렬(합집합)', alerts_parallel, False),
         ('병렬(순위융합)', alerts_fusion, False))

t0 = time.time(); rows = []
for tag in ('va', 'te'):
    c = CTX[tag]; sp = P9[tag][:, :8].max(1)
    oop_true = (c['y9'] == 8)                       # 정답 기준 '패턴 외' — 오라클 라우팅
    for head in ('L_oop', 'L_all'):
        sl = HEADS[head][tag]
        for arch, fn, use_oracle in ARCHS:
            for K in BUDGETS:
                for fr in KP_FRACS:
                    kp = int(round(K * fr)); kl = K - kp
                    ip, il = fn(sp, sl, kp, kl, oop_true if use_oracle else None)
                    r = eval_alerts(ip, il, c['ispos'], c['y9'], c['src'], c['dst'],
                                    c['ts'], c['att'], c['elig'])
                    r.update({'split': tag, 'head': head, 'arch': arch, 'K': K, 'kp_frac': fr})
                    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'serial_vs_parallel.csv', index=False)
print(f'{len(df):,}칸 · {time.time() - t0:,.0f}s')

채점 자격 시도 105건 / 전체 363건


960칸 · 43s


In [13]:
# val 에서 각 칸(머리×구조×예산)의 최선 kp_frac 을 고른다
va = df[df.split == 'va']
pick = (va.loc[va.groupby(['head', 'arch', 'K'])['f1'].idxmax()]
          [['head', 'arch', 'K', 'kp_frac', 'f1', 'precision', 'recall']])
print('=== val 에서 고른 운영점 (F1 기준) ===')
display(pick.round(4).reset_index(drop=True))

# test 는 그 운영점에서 딱 한 번
te = df[df.split == 'te'].set_index(['head', 'arch', 'K', 'kp_frac'])
fin = []
for _, p in pick.iterrows():
    r = te.loc[(p['head'], p['arch'], int(p['K']), float(p['kp_frac']))].to_dict()
    r.update({'head': p['head'], 'arch': p['arch'], 'K': int(p['K']), 'kp_frac': p['kp_frac']})
    fin.append(r)
fdf = pd.DataFrame(fin)
fdf.to_csv(OUT / 'serial_vs_parallel_test.csv', index=False)

print('\n=== test · F1 (val 로 고른 운영점에서 한 번) ===')
display(fdf.pivot_table(index=['head', 'arch'], columns='K', values='f1').round(4))
for col in ('precision', 'recall', 'block_precision', 'attempt_detection'):
    print(f'-- {col} --')
    display(fdf.pivot_table(index=['head', 'arch'], columns='K', values=col).round(4))

=== val 에서 고른 운영점 (F1 기준) ===


,head,arch,K,kp_frac,f1,precision,recall
0,L_all,병렬(순위융합),300,0.4,0.3572,0.8233,0.2281
1,L_all,병렬(순위융합),500,0.7,0.4826,0.7640,0.3527
2,L_all,병렬(순위융합),843,0.5,0.5296,0.6050,0.4709
3,L_all,병렬(순위융합),1200,0.8,0.5134,0.4883,0.5411
4,L_all,병렬(순위융합),2000,0.6,0.4204,0.3240,0.5983
5,L_all,병렬(순위융합),3000,0.2,0.3375,0.2297,0.6362
6,L_all,병렬(합집합),300,1.0,0.3413,0.7867,0.2179
7,L_all,병렬(합집합),500,1.0,0.4637,0.7340,0.3389
8,L_all,병렬(합집합),843,1.0,0.5213,0.5955,0.4635
9,L_all,병렬(합집합),1200,1.0,0.5028,0.4783,0.5300



=== test · F1 (val 로 고른 운영점에서 한 번) ===


K                   300     500     843     1200    2000    3000
head  arch                                                      
L_all 병렬(순위융합)    0.3382  0.4577  0.5065  0.4848  0.4066  0.3268
      병렬(합집합)     0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬          0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬(오라클라우팅)  0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
L_oop 병렬(순위융합)    0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      병렬(합집합)     0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬          0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬(오라클라우팅)  0.3313  0.4504  0.4894  0.4789  0.3990  0.3263

-- precision --


...

K                   300     500     843     1200    2000    3000
head  arch                                                      
L_all 병렬(순위융합)    0.2135  0.3290  0.4401  0.4969  0.5591  0.5923
      병렬(합집합)     0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
      직렬          0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
      직렬(오라클라우팅)  0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
L_oop 병렬(순위융합)    0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
      병렬(합집합)     0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
      직렬          0.2091  0.3237  0.4252  0.4908  0.5486  0.5914
      직렬(오라클라우팅)  0.2091  0.3237  0.4252  0.4908  0.5486  0.5914

-- block_precision --


K                   300     500     843     1200    2000    3000
head  arch                                                      
L_all 병렬(순위융합)    0.8772  0.7859  0.7582  0.5327  0.4617  0.6720
      병렬(합집합)     0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
      직렬          0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
      직렬(오라클라우팅)  0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
L_oop 병렬(순위융합)    0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
      병렬(합집합)     0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
      직렬          0.7958  0.7340  0.5659  0.4573  0.3130  0.2293
      직렬(오라클라우팅)  0.7958  0.7340  0.5659  0.4573  0.3130  0.2293

-- attempt_detection --


K                   300     500   843   1200  2000  3000
head  arch                                              
L_all 병렬(순위융합)    0.5556  0.8889   1.0   1.0   1.0   1.0
      병렬(합집합)     0.6667  0.8889   1.0   1.0   1.0   1.0
      직렬          0.6667  0.8889   1.0   1.0   1.0   1.0
      직렬(오라클라우팅)  0.6667  0.8889   1.0   1.0   1.0   1.0
L_oop 병렬(순위융합)    0.6667  0.8889   1.0   1.0   1.0   1.0
      병렬(합집합)     0.6667  0.8889   1.0   1.0   1.0   1.0
      직렬          0.6667  0.8889   1.0   1.0   1.0   1.0
      직렬(오라클라우팅)  0.6667  0.8889   1.0   1.0   1.0   1.0

## 11. 저장

1차 모델을 재학습한 경우에만 모델 객체를 저장한다(캐시 모드에서는 확률만 있으므로 생략).
`score_recipe` 를 함께 박아둔다 — **이 순서를 바꾸면 `tau` 가 무의미해진다.**

In [14]:
SCORE_RECIPE = (
    '1) X 를 cols 순서로 자른다  '
    '2) p = model.predict_proba(X) (9열)  '
    '3) **사전확률 보정을 반드시 적용한다**: p~ ∝ p * (prior_true / prior_model), 행별 재정규화  '
    '4) score = p~[:, :8].max(1), type = p~[:, :8].argmax(1)  '
    '5) score >= tau 이면 즉시 알림(유형 = type), 아니면 클래스 8 로 2차 라우팅  '
    '(3번을 빼면 tau 가 맞지 않는다. 보정 여부와 tau 는 한 묶음이다.)')

summary = {
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'basis': RUN['BASIS'], 'stage1_config': CFG,
    'operating_point_from_val': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                                 for k, v in op.items()},
    'score_recipe': SCORE_RECIPE,
    'test_stage1': {k: {kk: (None if isinstance(vv, float) and np.isnan(vv) else vv)
                        for kk, vv in v.items()} for k, v in res.items()},
    'arch_note': '직렬 vs 병렬은 팀 미결. 이 노트북은 네 구조를 재기만 하고 고르지 않는다.',
    'refit': {k: RUN[k] for k in ('REFIT_STAGE1', 'REFIT_BINARY')},
}
(OUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str),
                                  encoding='utf-8')

if RUN['REFIT_STAGE1'] and m9 is not None:
    import joblib
    joblib.dump({'model': m9, 'cols': tr['cols'],
                 'feat_names': [b.feat_names[c] for c in tr['cols']],
                 'config': CFG, 'basis': RUN['BASIS'], 'tau': TAU,
                 'prior_model': tr['prior_model'].tolist(),
                 'prior_true': tr['prior_true'].tolist(),
                 'score_recipe': SCORE_RECIPE}, OUT / 'stage1_9class.joblib')
    print('모델 저장:', OUT / 'stage1_9class.joblib')

print('산출물:', OUT)
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:<32}{p.stat().st_size / 1e6:>10,.2f} MB')

산출물: /workspace/model_pipeline
  serial_vs_parallel.csv                0.17 MB
  serial_vs_parallel_test.csv           0.01 MB
  summary.json                          0.00 MB


---
## 남은 미결 (이 노트북이 정하지 않은 것)

1. **직렬 vs 병렬** — 네 구조를 재기만 했다. 08-31 보고서에 남은 공정성 문제(병렬이 중복 제거 후
   예산을 다 못 채워 불리했음)를 먼저 닫아야 결론을 낼 수 있다.
2. **어느 이진 머리를 운영에 쓸지** — `L_oop` 는 오라클 라우팅으로 만든 학습셋이라 직렬에
   유리하게 편향돼 있다. 편향을 안고 있는 채로 비교값만 낸 상태다.
3. **알림 예산 K** — 실제 관제 인원이 하루에 몇 건을 볼 수 있는지는 팀이 정하지 않았다.
   지금은 6개 값을 나란히 낸다.
4. **정밀도 90% 목표** — val 에서 달성 가능한 지점과 그때의 재현율을 §7 이 그대로 출력한다.
   목표를 유지할지 조정할지는 팀 결정이다.

## test 사용 규율

이 노트북은 test 를 여러 번 예측하지만 **어느 수치도 선택에 되먹이지 않는다.**
다만 test 를 여러 각도로 들여다본 것이므로, 앞으로 이 test 로 새 설정을 고르면 그 수치는 못 쓴다.